In [2]:
!pip install boto3



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 79.2 MB/s eta 0:00:0000:01


In [3]:
import io
import zipfile
import boto3
import pandas as pd

# Connect to MinIO
s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="quantum",
    aws_secret_access_key="quantum-course-only",
    region_name="us-east-1",
)


In [8]:
import hashlib
import os
import re

# Ensure output directories exist
os.makedirs("silver/google_qec", exist_ok=True)
os.makedirs("results/part1", exist_ok=True)

#recreate table each time 
empty_trace = pd.DataFrame(columns=[
    "source_record_id", "source_name", "bronze_object",
    "archive_member", "record_locator", "input_sha256"
])
empty_trace.to_parquet("results/part1/source_trace.parquet")

# Stream Google QEC Bronze zip
key = "bronze/source=google_qec/google-surface-code-curated.zip"
obj = s3.get_object(Bucket="quantum-lake", Key=key)
raw_bytes = obj["Body"].read()
# version of the zip of goole_surface
archive_sha = hashlib.sha256(raw_bytes).hexdigest()

exp_rows = []
shot_rows = []
trace_rows = []
issue_rows = []

with zipfile.ZipFile(io.BytesIO(raw_bytes)) as z:
    all_members = z.namelist()

    # Discover unique experiment directories
    #(surface_code_bX_d3_r25_center
    exp_dirs = sorted(
        list(
            {
                m.split("/")[0]
                for m in all_members
                if "/" in m and not m.startswith("__")
            }
        )
    )

    for exp_dir in exp_dirs:
        # Parse folder name parameters (e.g. surface_code_bX_d3_r25_center_3_5)
        # Regex to capture: basis, distance, rounds, center_row, center_col
        match = re.search(
            r"surface_code_b([XZ])_d(\d+)_r(\d+)_center_(\d+)_(\d+)", exp_dir
        )
        if not match:
            continue

        basis, distance, rounds, c_row, c_col = match.groups()
        exp_id = f"google_{exp_dir}"
        exp_rec_id = (
            f"exp_{hashlib.md5(exp_id.encode()).hexdigest()[:12]}"
        ) # experiment id

        b8_member = f"{exp_dir}/detection_events.b8"
        label_member = f"{exp_dir}/obs_flips_actual.01"
        measurements_member = f"{exp_dir}/measurements.b8" #
        # missing files defense
        if b8_member not in all_members or label_member not in all_members:
            issue_rows.append(
                {
                    "source_name": "google_qec",
                    "bronze_object": key,
                    "archive_member": exp_dir,
                    "issue_type": "MISSING_COMPANION_FILE",
                    "details": "Missing detection_events.b8 or obs_flips_actual.01",
                }
            )
            continue

        # Read companion files
        #(surface_code_bX_d3_r25_center_3_5/detection_events.b8)
        b8_bytes = z.read(b8_member)
        meas_bytes = z.read(measurements_member)   #
        label_lines = [
            line.strip()
            for line in z.read(label_member).decode("utf-8").splitlines()
            if line.strip()
        ]

        num_shots = len(label_lines)
        bytes_per_shot = (
            len(b8_bytes) // num_shots if num_shots > 0 else 0
        )

        meas_bytes_per_shot = (  #                                    # <-- ADD THIS BLOCK
            len(meas_bytes) // num_shots if num_shots > 0 else 0
        )
        # Validate alignment
        #SHARED TABLE
        if len(b8_bytes) != bytes_per_shot * num_shots:
            issue_rows.append(
                {
                    "source_name": "google_qec",
                    "bronze_object": key,
                    "archive_member": exp_dir,
                    "issue_type": "ALIGNMENT_MISMATCH",
                    "details": f"b8 byte length {len(b8_bytes)} not divisible by shot count {num_shots}",
                }
            )

        # Add Experiment metadata record
        #TABLE 2
        exp_rows.append(
            {
                "source_record_id": exp_rec_id,
                "experiment_id": exp_id,
                "basis": basis,
                "distance": int(distance),
                "rounds": int(rounds),
                "shots": num_shots,
                "center_row": int(c_row),
                "center_col": int(c_col),
                "measurement_count": meas_bytes_per_shot * 8,
                "detector_count": bytes_per_shot * 8,
            }
        )

        trace_rows.append(
            {
                "source_record_id": exp_rec_id,
                "source_name": "google_qec",
                "bronze_object": key,
                "archive_member": exp_dir,
                "record_locator": "experiment_metadata",
                "input_sha256": archive_sha,
            }
        )

        # Process individual aligned shots
        #TABLE 3 
        for shot_idx in range(num_shots):
            shot_rec_id = f"google_shot_{hashlib.md5(f'{exp_id}_{shot_idx}'.encode()).hexdigest()[:12]}"
            start_b = shot_idx * bytes_per_shot
            shot_b8 = b8_bytes[start_b : start_b + bytes_per_shot]

            start_m = shot_idx * meas_bytes_per_shot 
            shot_meas = meas_bytes[start_m : start_m + meas_bytes_per_shot] 
            # Count set bits in detector bytes
            event_cnt = bin(
                int.from_bytes(shot_b8, byteorder="little")
            ).count("1")

            shot_rows.append(
                {
                    "source_record_id": shot_rec_id,
                    "experiment_id": exp_id,
                    "shot_index": shot_idx,
                    "measurement_bits": shot_meas,
                    "sweep_bits": b"",
                    "detector_bits": shot_b8,
                    "detector_event_count": event_cnt,
                    "actual_observable_flip": bool(
                        int(label_lines[shot_idx])
                    ),
                    "belief_matching_prediction": False,
                    "correlated_matching_prediction": False,
                    "pymatching_prediction": False,
                    "tensor_network_contraction_prediction": False,
                }
            )

            trace_rows.append(
                {
                    "source_record_id": shot_rec_id,
                    "source_name": "google_qec",
                    "bronze_object": key,
                    "archive_member": f"{b8_member} + {label_member}",
                    "record_locator": f"shot_index_{shot_idx}",
                    "input_sha256": archive_sha,
                }
            )

# Save Google QEC Silver Tables
df_exp = pd.DataFrame(exp_rows)
display(df_exp)
df_shots = pd.DataFrame(shot_rows)
df_issues = pd.DataFrame(issue_rows)

df_exp.to_parquet("silver/google_qec/experiment.parquet")
df_shots.to_parquet("silver/google_qec/shot.parquet")

# Append new trace rows to existing source_trace.parquet
df_trace_existing = pd.read_parquet("results/part1/source_trace.parquet")
df_trace_updated = pd.concat(
    [df_trace_existing, pd.DataFrame(trace_rows)], ignore_index=True
)
df_trace_updated.to_parquet("results/part1/source_trace.parquet")

if not df_issues.empty:
    df_issues.to_parquet("results/part1/data_issues.parquet")

print("=== Google QEC Silver Pipeline Completed ===")
print(f"Experiments Processed: {len(df_exp):,}")
print(f"Shots Processed: {len(df_shots):,}")
print(f"Total Source Trace Rows: {len(df_trace_updated):,}")
print(f"Quarantined Issues: {len(df_issues):,}")

,source_record_id,experiment_id,basis,distance,rounds,shots,center_row,center_col,measurement_count,detector_count
0,exp_dc1c8c81dc52,google_surface_code_bX_d3_r25_center_3_5,X,3,25,50000,3,5,216,200
1,exp_b3206f3b7bf7,google_surface_code_bX_d3_r25_center_5_3,X,3,25,50000,5,3,216,200
2,exp_2cee3fad4f1c,google_surface_code_bX_d3_r25_center_5_7,X,3,25,50000,5,7,216,200
3,exp_a644e3a2d8d6,google_surface_code_bX_d3_r25_center_7_5,X,3,25,50000,7,5,216,200
4,exp_fbfa44cc439a,google_surface_code_bX_d5_r25_center_5_5,X,5,25,50000,5,5,632,600


=== Google QEC Silver Pipeline Completed ===
Experiments Processed: 5
Shots Processed: 250,000
Total Source Trace Rows: 250,005
Quarantined Issues: 0
